# Two-Stage Cascade Evaluation

Phase 2 Isolation Forest packet alerts are re-checked with Phase 3 XGBoost on linked flows.

Run the pipeline first:

```bash
python scripts/run_two_stage_cascade.py --seed 1
```


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
RESULTS = PROJECT_ROOT / "reports" / "cascade"

metrics = pd.read_csv(RESULTS / "cascade_metrics.csv")
per_attack = pd.read_csv(RESULTS / "cascade_per_attack.csv")
summary = json.loads((RESULTS / "cascade_summary.json").read_text())

display(metrics.round(4))
print(f"FP reduction: {summary['false_positive_reduction_pct']:.1f}%")
print(f"Alert→flow link rate: {summary['alert_flow_link_rate']:.1%}")


In [ ]:
compare = metrics[metrics["model"].isin(["Phase2-Only-Test", "TwoStage-Cascade-Test"])].copy()
compare = compare.set_index("model")[["precision", "recall", "f1", "fpr"]]

ax = compare.T.plot(kind="bar", figsize=(8, 4))
ax.set_ylabel("Score")
ax.set_title("Phase 2 vs Two-Stage Cascade (Test)")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(loc="best")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

display(compare.round(4))


In [ ]:
plot_df = per_attack.set_index("attack_type")[
    ["phase2_detection_rate", "cascade_detection_rate"]
] * 100

ax = plot_df.plot(kind="barh", figsize=(9, 5))
ax.set_xlabel("Detection rate (%)")
ax.set_title("Per-attack detection: Phase 2 vs Cascade")
ax.legend(["Phase 2", "Cascade"])
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

display(per_attack.round(4))


## Interpretation

- The cascade keeps flood attacks (DDoS/DoS HTTP) nearly unchanged while cutting false positives sharply.
- DNS Spoofing / Brute Force / XSS remain hard at packet level; flow re-checking removes many benign false alarms but can also drop some weak attack alerts.
- Report the FP-reduction percentage and the precision/recall trade-off as the main Phase 3 contribution.
